In [51]:
import importlib
import fetch_data as td
import model as mod

In [75]:
# Après avoir modifié le fichier .py :
importlib.reload(mod)

<module 'model' from '/home/onyxia/MiseEnProd/model.py'>

In [2]:
ids = td.get_movie_ids_list(td.headers, 50, ending_date="03-18-2026", ascending=False)
historic_df = td.get_movies_info(ids, td.headers)
df_cleaned = td.clean_data(historic_df)
df_cleaned.head()

getting movie ids


  0%|          | 0/50 [00:00<?, ?it/s]

100%|██████████| 50/50 [00:05<00:00,  8.41it/s]


getting movie info


100%|██████████| 1000/1000 [03:47<00:00,  4.39it/s]


,belongs_to_collection,budget,id,origin_country,overview,popularity,release_date,revenue,runtime,title,vote_average,vote_count,main_genre_id,main_genre_name,full_poster_path,overview_count,title_count,timestamp
0,NaN,150000000,1327819,US,Scientists have discovered how to 'hop' human ...,137.5015,2026-03-04,164700900,105,Hoppers,7.735,215,16,Animation,https://image.tmdb.org/t/p/original//xjtWQ2CL1...,296,7,1772582400
1,NaN,80000000,1159831,US,A lonely Frankenstein travels to 1930s Chicago...,36.0001,2026-03-04,21000000,126,The Bride!,6.282,172,878,Science Fiction,https://image.tmdb.org/t/p/original//lV8YHwGkY...,242,10,1772582400
2,NaN,45000000,1159559,US,When a new Ghostface killer emerges in the qui...,130.1839,2026-02-25,177977863,114,Scream 7,5.900,349,27,Horror,https://image.tmdb.org/t/p/original//jjyuk0edL...,293,8,1771977600
3,NaN,0,799882,US,When her tranquil life on a remote island is s...,173.2240,2026-02-17,0,102,The Bluff,6.313,283,28,Action,https://image.tmdb.org/t/p/original//w0jwjFmRu...,223,9,1771286400
4,NaN,20000000,1119449,US,A 'Man from the Future' arrives at an LA diner...,78.6582,2026-02-13,8705352,135,"Good Luck, Have Fun, Don't Die",6.900,182,878,Science Fiction,https://image.tmdb.org/t/p/original//rWcfOdY7T...,227,30,1770940800


In [3]:
df_train = df_cleaned[df_cleaned['revenue']>0]
y = df_train['revenue']

In [4]:
pipeline = mod.create_random_forest_pipeline()
scores = mod.model_cross_validation(df_train, pipeline)

In [5]:
print("Moyenne du revenue    :", y.mean())
print("Écart-type du revenue :", y.std())
print("Médiane du revenue    :", y.median())

# RMSE relatif (coefficient de variation)
rmse_moyen = - scores.mean()
print("RMSE :", rmse_moyen)
print("RMSE relatif (RMSE / std)  :", rmse_moyen / y.std())
print("RMSE relatif (RMSE / mean) :", rmse_moyen / y.mean())

Moyenne du revenue    : 88684860.26824817
Écart-type du revenue : 221556700.71611667
Médiane du revenue    : 14339357.5
RMSE : 152361845.19540375
RMSE relatif (RMSE / std)  : 0.6876878230400572
RMSE relatif (RMSE / mean) : 1.7180141541019471


In [9]:
param_grid = {
    "n_estimators":       [100],
    "max_depth":          [None],
    "tfidf_max_features": [3000],
    "tfidf_ngram_range":  [(1,1), (1,2), (2,3)],
    "tfidf_min_df":       [2, 5],
}

In [10]:
results = mod.grid_search_random_forest(df_train, param_grid=param_grid)
results

6 combinaisons à tester...


RMSE: 149,929,111 (+/- 72,811,400) | {'n_estimators': 100, 'max_depth': None, 'tfidf_max_features': 3000, 'tfidf_ngram_range': (1, 1), 'tfidf_min_df': 2}
RMSE: 150,517,516 (+/- 71,832,780) | {'n_estimators': 100, 'max_depth': None, 'tfidf_max_features': 3000, 'tfidf_ngram_range': (1, 1), 'tfidf_min_df': 5}
RMSE: 148,824,499 (+/- 75,020,628) | {'n_estimators': 100, 'max_depth': None, 'tfidf_max_features': 3000, 'tfidf_ngram_range': (1, 2), 'tfidf_min_df': 2}
RMSE: 148,269,571 (+/- 74,590,428) | {'n_estimators': 100, 'max_depth': None, 'tfidf_max_features': 3000, 'tfidf_ngram_range': (1, 2), 'tfidf_min_df': 5}
RMSE: 146,415,088 (+/- 79,385,113) | {'n_estimators': 100, 'max_depth': None, 'tfidf_max_features': 3000, 'tfidf_ngram_range': (2, 3), 'tfidf_min_df': 2}
RMSE: 149,518,541 (+/- 77,878,272) | {'n_estimators': 100, 'max_depth': None, 'tfidf_max_features': 3000, 'tfidf_ngram_range': (2, 3), 'tfidf_min_df': 5}


,n_estimators,max_depth,tfidf_max_features,tfidf_ngram_range,tfidf_min_df,rmse_mean,rmse_std
4,100,None,3000,"(2, 3)",2,1.464151e+08,7.938511e+07
3,100,None,3000,"(1, 2)",5,1.482696e+08,7.459043e+07
2,100,None,3000,"(1, 2)",2,1.488245e+08,7.502063e+07
5,100,None,3000,"(2, 3)",5,1.495185e+08,7.787827e+07
0,100,None,3000,"(1, 1)",2,1.499291e+08,7.281140e+07
1,100,None,3000,"(1, 1)",5,1.505175e+08,7.183278e+07


In [56]:
results = mod.grid_search_elastic_net(df_train)

576 combinaisons à tester...
RMSE: 145,377,964 (+/- 59,770,446) | {'alpha': 0.01, 'l1_ratio': 0.1, 'tfidf_max_features': 1000, 'tfidf_ngram_range': (1, 1), 'tfidf_min_df': 1}
RMSE: 145,205,324 (+/- 59,787,858) | {'alpha': 0.01, 'l1_ratio': 0.1, 'tfidf_max_features': 1000, 'tfidf_ngram_range': (1, 1), 'tfidf_min_df': 2}
RMSE: 145,021,941 (+/- 59,408,912) | {'alpha': 0.01, 'l1_ratio': 0.1, 'tfidf_max_features': 1000, 'tfidf_ngram_range': (1, 1), 'tfidf_min_df': 5}
RMSE: 144,819,931 (+/- 59,606,791) | {'alpha': 0.01, 'l1_ratio': 0.1, 'tfidf_max_features': 1000, 'tfidf_ngram_range': (1, 2), 'tfidf_min_df': 1}
RMSE: 144,747,928 (+/- 59,628,939) | {'alpha': 0.01, 'l1_ratio': 0.1, 'tfidf_max_features': 1000, 'tfidf_ngram_range': (1, 2), 'tfidf_min_df': 2}
RMSE: 144,792,540 (+/- 59,684,531) | {'alpha': 0.01, 'l1_ratio': 0.1, 'tfidf_max_features': 1000, 'tfidf_ngram_range': (1, 2), 'tfidf_min_df': 5}
RMSE: 144,708,611 (+/- 60,744,835) | {'alpha': 0.01, 'l1_ratio': 0.1, 'tfidf_max_features': 300

In [81]:
param_grid = {
            "alpha":             [0.01, 0.1, 1.0],
            "l1_ratio":          [0.5, 0.7, 0.9],
            "tfidf_max_features":[5000, 10000],
            "tfidf_ngram_range": [(1,1), (1,2)],
            "tfidf_min_df":      [2, 5, 10, 20, 30],
            "tfidf_max_df":      [0.6, 0.8, 0.9]
        }

In [82]:
results_new = mod.grid_search_elastic_net(df_train, param_grid)

540 combinaisons à tester...
RMSE: 146,919,867 (+/- 59,773,628) | {'alpha': 0.01, 'l1_ratio': 0.5, 'tfidf_max_features': 5000, 'tfidf_ngram_range': (1, 1), 'tfidf_min_df': 2, 'tfidf_max_df': 0.6}
RMSE: 146,919,867 (+/- 59,773,628) | {'alpha': 0.01, 'l1_ratio': 0.5, 'tfidf_max_features': 5000, 'tfidf_ngram_range': (1, 1), 'tfidf_min_df': 2, 'tfidf_max_df': 0.8}
RMSE: 146,919,867 (+/- 59,773,628) | {'alpha': 0.01, 'l1_ratio': 0.5, 'tfidf_max_features': 5000, 'tfidf_ngram_range': (1, 1), 'tfidf_min_df': 2, 'tfidf_max_df': 0.9}
RMSE: 148,003,460 (+/- 58,291,009) | {'alpha': 0.01, 'l1_ratio': 0.5, 'tfidf_max_features': 5000, 'tfidf_ngram_range': (1, 1), 'tfidf_min_df': 5, 'tfidf_max_df': 0.6}
RMSE: 148,003,460 (+/- 58,291,009) | {'alpha': 0.01, 'l1_ratio': 0.5, 'tfidf_max_features': 5000, 'tfidf_ngram_range': (1, 1), 'tfidf_min_df': 5, 'tfidf_max_df': 0.8}
RMSE: 148,003,460 (+/- 58,291,009) | {'alpha': 0.01, 'l1_ratio': 0.5, 'tfidf_max_features': 5000, 'tfidf_ngram_range': (1, 1), 'tfidf_mi

In [83]:
results_new.sort_values(["rmse_mean"]).head(20)

,alpha,l1_ratio,tfidf_max_features,tfidf_ngram_range,tfidf_min_df,tfidf_max_df,rmse_mean,rmse_std
266,0.1,0.7,5000,"(1, 2)",20,0.9,1.428925e+08,6.302261e+07
264,0.1,0.7,5000,"(1, 2)",20,0.6,1.428925e+08,6.302261e+07
265,0.1,0.7,5000,"(1, 2)",20,0.8,1.428925e+08,6.302261e+07
250,0.1,0.7,5000,"(1, 1)",20,0.8,1.428925e+08,6.302261e+07
251,0.1,0.7,5000,"(1, 1)",20,0.9,1.428925e+08,6.302261e+07
249,0.1,0.7,5000,"(1, 1)",20,0.6,1.428925e+08,6.302261e+07
280,0.1,0.7,10000,"(1, 1)",20,0.8,1.428925e+08,6.302261e+07
279,0.1,0.7,10000,"(1, 1)",20,0.6,1.428925e+08,6.302261e+07
281,0.1,0.7,10000,"(1, 1)",20,0.9,1.428925e+08,6.302261e+07
295,0.1,0.7,10000,"(1, 2)",20,0.8,1.428925e+08,6.302261e+07
